# ============================================
# GLAUCOMA DETECTION — Vision Transformer (ViT)
# ============================================

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import json
from collections import Counter
from sklearn.metrics import classification_report

## Mount Google Drive (Only if using Google Colab)

In [3]:
# -------------------------------------------------------
# ONLY run this cell if you are on Google Colab
# Comment it out if you are running on a local machine
# -------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. CONFIGURATION — Set Your Paths Here

In [4]:
# ============================================================
#  CHOOSE YOUR ENVIRONMENT — uncomment the one that applies
# ============================================================

# --- OPTION A: Google Colab (your original paths) ---
TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/glaucoma_dataset/dataset/train"
TEST_PATH  = "/content/drive/MyDrive/Colab Notebooks/glaucoma_dataset/dataset/test"
SAVE_DIR   = "/content/drive/MyDrive/Colab Notebooks/glaucoma_models_vit"

# --- OPTION B: Local Machine — Windows example ---
# TRAIN_PATH = r"C:\Users\YourName\Documents\glaucoma_dataset\dataset\train"
# TEST_PATH  = r"C:\Users\YourName\Documents\glaucoma_dataset\dataset\test"
# SAVE_DIR   = r"C:\Users\YourName\Documents\glaucoma_models_vit"

# --- OPTION C: Local Machine — Linux / Mac example ---
# TRAIN_PATH = "/home/yourname/datasets/glaucoma_dataset/dataset/train"
# TEST_PATH  = "/home/yourname/datasets/glaucoma_dataset/dataset/test"
# SAVE_DIR   = "/home/yourname/datasets/glaucoma_models_vit"

# ============================================================
# NOTE: Your dataset folder structure should look like this:
#
#   train/
#     glaucoma/
#       img1.jpg
#       img2.jpg
#     normal/
#       img1.jpg
#       img2.jpg
#
#   test/
#     glaucoma/
#       ...
#     normal/
#       ...
# ============================================================

os.makedirs(SAVE_DIR, exist_ok=True)

# Training hyperparameters
BATCH_SIZE    = 32
EPOCHS        = 1
LEARNING_RATE = 0.0001   # ViT fine-tuning works best with a smaller lr than ResNet
IMAGE_SIZE    = 224       # ViT-B/16 expects 224x224

print("Training Configuration")
print(f"  Train path  : {TRAIN_PATH}")
print(f"  Test path   : {TEST_PATH}")
print(f"  Save dir    : {SAVE_DIR}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Epochs      : {EPOCHS}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  Image size  : {IMAGE_SIZE}x{IMAGE_SIZE}")

Training Configuration
  Train path  : /content/drive/MyDrive/Colab Notebooks/glaucoma_dataset/dataset/train
  Test path   : /content/drive/MyDrive/Colab Notebooks/glaucoma_dataset/dataset/test
  Save dir    : /content/drive/MyDrive/Colab Notebooks/glaucoma_models_vit
  Batch size  : 32
  Epochs      : 1
  LR          : 0.0001
  Image size  : 224x224


## 2. DATA LOADING & AUGMENTATION

In [5]:
# ViT is more sensitive to input quality, so we use slightly
# stronger augmentation than ResNet to improve generalization

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Loading datasets...")
train_dataset = datasets.ImageFolder(root=TRAIN_PATH, transform=train_transform)
test_dataset  = datasets.ImageFolder(root=TEST_PATH,  transform=test_transform)

class_names = train_dataset.classes
print(f"  Classes found    : {class_names}")
print(f"  Training samples : {len(train_dataset)}")
print(f"  Test samples     : {len(test_dataset)}")

Loading datasets...
  Classes found    : ['glaucoma', 'normal']
  Training samples : 17830
  Test samples     : 774


In [6]:
 # Handle class imbalance with weights
train_counts  = Counter([label for _, label in train_dataset])
print("Class distribution:")
for i, name in enumerate(class_names):
    print(f"  {name}: {train_counts[i]} images")

class_weights = 1.0 / torch.tensor([train_counts[i] for i in range(len(class_names))], dtype=torch.float)
class_weights = class_weights / class_weights.sum()
print(f"\nClass weights: {class_weights}")

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Class distribution:
  glaucoma: 6950 images
  normal: 10880 images

Class weights: tensor([0.6102, 0.3898])


## 3. BUILD THE ViT MODEL

In [7]:
print("Building Vision Transformer (ViT-B/16) model...")

# Load pretrained ViT-B/16 from torchvision
# ViT-B/16 = Base model, patch size 16x16, trained on ImageNet
model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

# -------------------------------------------------------
# FREEZING STRATEGY
# Freeze all layers first, then selectively unfreeze
# -------------------------------------------------------
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last 4 transformer encoder blocks for fine-tuning
# ViT-B/16 has 12 encoder blocks (encoder.layers.0 ... encoder.layers.11)
num_blocks = len(model.encoder.layers)
unfreeze_from = num_blocks - 4  # unfreeze last 4 blocks

for i in range(unfreeze_from, num_blocks):
    for param in model.encoder.layers[i].parameters():
        param.requires_grad = True

# Also unfreeze the LayerNorm at the end of the encoder
for param in model.encoder.ln.parameters():
    param.requires_grad = True

# -------------------------------------------------------
# REPLACE THE CLASSIFICATION HEAD
# In ViT, the head is at model.heads.head (not model.fc)
# -------------------------------------------------------
in_features = model.heads.head.in_features  # 768 for ViT-B/16

model.heads = nn.Sequential(
    nn.LayerNorm(in_features),
    nn.Linear(in_features, 256),
    nn.GELU(),
    nn.Dropout(0.4),
    nn.Linear(256, len(class_names))
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"  Model         : ViT-B/16")
print(f"  Device        : {device}")
if device.type == 'cuda':
    print(f"  GPU           : {torch.cuda.get_device_name(0)}")
print(f"  Total params  : {total/1e6:.1f}M")
print(f"  Trainable     : {trainable/1e6:.1f}M  (frozen: {(total-trainable)/1e6:.1f}M)")

Building Vision Transformer (ViT-B/16) model...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:03<00:00, 109MB/s]


  Model         : ViT-B/16
  Device        : cpu
  Total params  : 86.0M
  Trainable     : 28.6M  (frozen: 57.4M)


## 4. LOSS, OPTIMIZER & SCHEDULER

In [8]:
# Loss with class weights to handle imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# AdamW works better than Adam for ViT (weight decay regularization)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LEARNING_RATE, weight_decay=0.01)

# Cosine annealing scheduler — smoothly reduces LR over epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print("Training components:")
print(f"  Loss       : CrossEntropyLoss with class weights")
print(f"  Optimizer  : AdamW  (lr={LEARNING_RATE}, weight_decay=0.01)")
print(f"  Scheduler  : CosineAnnealingLR")

Training components:
  Loss       : CrossEntropyLoss with class weights
  Optimizer  : AdamW  (lr=0.0001, weight_decay=0.01)
  Scheduler  : CosineAnnealingLR


## 5. TRAINING LOOP

In [9]:
def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler,
                epochs, device, save_dir, class_names):

    train_losses, train_accuracies = [], []
    val_losses,   val_accuracies   = [], []
    best_val_acc = 0
    checkpoint   = None
    start_time   = time.time()

    print("\n" + "="*60)
    print("STARTING TRAINING  —  ViT-B/16")
    print("="*60)

    for epoch in range(epochs):

        # ===== TRAINING PHASE =====
        model.train()
        running_loss = 0
        correct = total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()

            # Gradient clipping — important for ViT stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            running_loss += loss.item()
            _, predicted  = torch.max(outputs, 1)
            total        += labels.size(0)
            correct      += (predicted == labels).sum().item()

            if (batch_idx + 1) % 20 == 0:
                print(f"  Epoch [{epoch+1}/{epochs}] "
                      f"Batch [{batch_idx+1}/{len(train_loader)}] "
                      f"Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(train_loader)
        epoch_acc  = 100 * correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)

        # ===== VALIDATION PHASE =====
        model.eval()
        val_loss = val_correct = val_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total   += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_epoch_loss = val_loss / len(test_loader)
        val_epoch_acc  = 100 * val_correct / val_total
        val_losses.append(val_epoch_loss)
        val_accuracies.append(val_epoch_acc)

        # Step the scheduler
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(f"\nEPOCH {epoch+1}/{epochs}")
        print(f"  Train  — Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.2f}%")
        print(f"  Valid  — Loss: {val_epoch_loss:.4f}  Acc: {val_epoch_acc:.2f}%")
        print(f"  LR     : {current_lr:.2e}")

        # Save best model
        if val_epoch_acc > best_val_acc:
            best_val_acc = val_epoch_acc
            checkpoint = {
                'epoch':                epoch + 1,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_accuracy':         best_val_acc,
                'class_names':          class_names,
                'model_architecture':   'vit_b_16'
            }
            best_model_path = os.path.join(save_dir, 'best_glaucoma_vit_model.pth')
            torch.save(checkpoint, best_model_path)
            print(f"  Best model saved! (Val Acc: {best_val_acc:.2f}%)")

        # Save latest checkpoint every epoch
        if checkpoint:
            torch.save(checkpoint, os.path.join(save_dir, 'latest_glaucoma_vit_model.pth'))

    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print(f"TRAINING COMPLETE")
    print(f"  Time taken          : {elapsed/60:.2f} minutes")
    print(f"  Best val accuracy   : {best_val_acc:.2f}%")
    print("="*60)

    return train_losses, train_accuracies, val_losses, val_accuracies, best_val_acc

## 6. START TRAINING

In [ ]:
train_losses, train_accuracies, val_losses, val_accuracies, best_acc = train_model(
    model        = model,
    train_loader = train_loader,
    test_loader  = test_loader,
    criterion    = criterion,
    optimizer    = optimizer,
    scheduler    = scheduler,
    epochs       = EPOCHS,
    device       = device,
    save_dir     = SAVE_DIR,
    class_names  = class_names
)


STARTING TRAINING  —  ViT-B/16


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Epoch [1/1] Batch [20/558] Loss: 0.5051
  Epoch [1/1] Batch [40/558] Loss: 0.7224
  Epoch [1/1] Batch [60/558] Loss: 0.8327
  Epoch [1/1] Batch [80/558] Loss: 0.5023
  Epoch [1/1] Batch [100/558] Loss: 0.4080
  Epoch [1/1] Batch [120/558] Loss: 0.3098
  Epoch [1/1] Batch [140/558] Loss: 0.3265
  Epoch [1/1] Batch [160/558] Loss: 0.4619
  Epoch [1/1] Batch [180/558] Loss: 0.4202
  Epoch [1/1] Batch [200/558] Loss: 0.4562
  Epoch [1/1] Batch [220/558] Loss: 0.4584
  Epoch [1/1] Batch [240/558] Loss: 0.3882
  Epoch [1/1] Batch [260/558] Loss: 0.2791
  Epoch [1/1] Batch [280/558] Loss: 0.5809
  Epoch [1/1] Batch [300/558] Loss: 0.3217
  Epoch [1/1] Batch [320/558] Loss: 0.4993
  Epoch [1/1] Batch [340/558] Loss: 0.5595
  Epoch [1/1] Batch [360/558] Loss: 0.4629
  Epoch [1/1] Batch [380/558] Loss: 0.5042
  Epoch [1/1] Batch [400/558] Loss: 0.2787
  Epoch [1/1] Batch [420/558] Loss: 0.6732
  Epoch [1/1] Batch [440/558] Loss: 0.2911
  Epoch [1/1] Batch [460/558] Loss: 0.4766
  Epoch [1/1] B

## 7. PLOT TRAINING RESULTS

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses,    label='Train Loss',     marker='o', linewidth=2)
ax1.plot(val_losses,      label='Validation Loss', marker='s', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss  —  ViT-B/16')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accuracies, label='Train Accuracy',     marker='o', linewidth=2)
ax2.plot(val_accuracies,   label='Validation Accuracy', marker='s', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy  —  ViT-B/16')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'vit_training_plots.png'))
plt.show()

## 8. FINAL EVALUATION

In [ ]:
print("\n" + "="*60)
print("FINAL MODEL EVALUATION")
print("="*60)

# Load best model
best_model_path = os.path.join(SAVE_DIR, 'best_glaucoma_vit_model.pth')
checkpoint      = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

correct = total = 0
all_predictions, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs        = model(images)
        _, predicted   = torch.max(outputs, 1)

        total   += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100 * correct / total
print(f"\nTest Accuracy: {test_accuracy:.2f}%")
print(f"\nDetailed Classification Report:")
print(classification_report(all_labels, all_predictions, target_names=class_names))

## 9. EXPORT MODEL FOR MOBILE

In [ ]:
print("\n" + "="*60)
print("EXPORTING MODEL FOR MOBILE APP")
print("="*60)

def export_for_mobile(model, class_names, save_dir, test_accuracy):
    model.eval()
    model.to('cpu')

    class MobileModel(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, x):
            return torch.softmax(self.model(x), dim=1)

    mobile_model  = MobileModel(model)
    mobile_model.eval()

    example_input = torch.randn(1, 3, 224, 224)
    traced_model  = torch.jit.trace(mobile_model, example_input)

    mobile_path = os.path.join(save_dir, 'glaucoma_vit_mobile_model.pt')
    traced_model.save(mobile_path)
    print(f"Mobile model saved : {mobile_path}")
    print(f"File size          : {os.path.getsize(mobile_path)/(1024*1024):.2f} MB")

    metadata = {
        'model_name':           'Glaucoma Detection Model',
        'architecture':         'vit_b_16',
        'input_shape':          [224, 224, 3],
        'normalization': {
            'mean': [0.485, 0.456, 0.406],
            'std':  [0.229, 0.224, 0.225]
        },
        'classes':              class_names,
        'num_classes':          len(class_names),
        'test_accuracy':        test_accuracy,
        'output_type':          'probability',
        'recommended_threshold': 0.7
    }

    metadata_path = os.path.join(save_dir, 'vit_model_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Metadata saved     : {metadata_path}")

    return mobile_path

export_for_mobile(model, class_names, SAVE_DIR, test_accuracy)

## 10. CONFIDENCE-BASED INFERENCE

In [ ]:
import random
from PIL import Image

# Load saved mobile model
mobile_model_path = os.path.join(SAVE_DIR, 'glaucoma_vit_mobile_model.pt')
mobile_model      = torch.jit.load(mobile_model_path)
mobile_model.eval()
print(f"Mobile model loaded from: {mobile_model_path}")


def predict_with_confidence(image_tensor, model, class_names, confidence_threshold=0.7):
    """
    Run inference and flag low-confidence predictions.

    Parameters
    ----------
    image_tensor        : torch.Tensor  shape (3, 224, 224) — already normalized
    model               : TorchScript mobile model
    class_names         : list of class name strings
    confidence_threshold: float — predictions below this are flagged

    Returns
    -------
    predicted_class, confidence, is_low_confidence, message
    """
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)   # add batch dim

    image_tensor = image_tensor.to('cpu')

    with torch.no_grad():
        probabilities = model(image_tensor)[0]     # softmax already applied

    max_prob, predicted_idx = torch.max(probabilities, 0)
    predicted_class  = class_names[predicted_idx.item()]
    confidence       = max_prob.item()
    is_low_confidence = confidence < confidence_threshold

    if is_low_confidence:
        message = (f"Low confidence ({confidence:.2f}). "
                   f"Image may be irrelevant or out-of-distribution. "
                   f"Please verify the image.")
    else:
        message = f"Prediction: {predicted_class}  (Confidence: {confidence:.2f})"

    return predicted_class, confidence, is_low_confidence, message


# --- Demo with a random test image ---
random_idx          = random.randint(0, len(test_dataset) - 1)
sample_tensor, label = test_dataset[random_idx]
actual_class        = class_names[label]

pred_class, conf, low_conf, msg = predict_with_confidence(
    sample_tensor, mobile_model, class_names, confidence_threshold=0.7
)

print(f"Actual class : {actual_class}")
print(f"Result       : {msg}")

# Visualize
inv_normalize = transforms.Normalize(
    mean=[-m/s for m, s in zip([0.485,0.456,0.406], [0.229,0.224,0.225])],
    std =[1/s   for s   in     [0.229,0.224,0.225]]
)
display_img = inv_normalize(sample_tensor).permute(1, 2, 0).numpy()
display_img = (display_img * 255).clip(0, 255).astype(np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(display_img)
plt.title(f"Predicted: {pred_class} ({conf:.2f})\nActual: {actual_class}")
plt.axis('off')
plt.show()